# 001_semantic_entity — 公司與別名對照表（建表＋灌種子）

- 用途：建 `s_semantic_entity`、`s_semantic_entity_alias`，從 [c05] 內嵌的種子資料 MERGE 進表。會議萃取 T1-c（參與者 side / company）、T2（customer 正規化）、T3（Sales Brief 品牌）共用。
- **正本是 [c05] `seed_data`**（隨本 notebook 進 git 留歷史）；表只是投影。**改對照 = 改 [c05] → 重跑本 notebook**。從 [c05] 拿掉的列在表中標 `is_active = false`，不刪。不需上傳任何檔案，整支 notebook 複製貼上即可。
- 不建 `b_` 層：幾十列人工維護資料，notebook 裡的種子就是原始層。
- 冪等：`CREATE TABLE IF NOT EXISTS` ＋ MERGE；重跑只更新有變的列。每次（含 dry run）寫一列 `b_meeting_migration_log`（dry run 不寫）。
- 參數（widgets）：`catalog`、`schema`、`dry_run`（預設 `true`：只驗證與預覽差異，不寫任何表）
- 負責人 / 更新日期：（填）/ 2026-10-14

| 表 | 鍵 | 內容 |
|---|---|---|
| `s_semantic_entity` | `entity_id` | 一家公司一列：名稱、`entity_type`（customer / self / supplier / partner / tool）、`side`（auo / customer / third_party / ignore）、`app_default` |
| `s_semantic_entity_alias` | `(alias_type, alias_norm)` | 一個別名一列：`alias_type` = `name`（素材原名）/ `email_domain`（後綴比對）/ `sb_brand`（Sales Brief `brand_code`） |
| `b_meeting_migration_log` | `(slug, run_at)` | migration 執行紀錄（§2.3） |

之後語意層計畫若開 `mi3_semantic` schema，在那邊建 View 指向這兩張表即可，不搬資料。

In [ ]:
# [c01] params
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
dry_run = dbutils.widgets.get("dry_run") == "true"
assert catalog and schema, "catalog / schema 不可為空"
print({"catalog": catalog, "schema": schema, "dry_run": dry_run})

In [ ]:
# [c02] imports
# 本 cell 與 [c03]～[c05] 不碰 spark / dbutils / 檔案，可被 tests/ 載入。
import csv
import io
import json
import re

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

SLUG = "001_semantic_entity"
ENTITY_COLS = ["entity_id", "entity_name", "entity_type", "side", "app_default", "note"]
ALIAS_COLS = ["alias_type", "alias", "entity_id", "note"]
ENTITY_TYPES = {"customer", "self", "supplier", "partner", "tool"}
SIDES = {"auo", "customer", "third_party", "ignore"}
ALIAS_TYPES = {"name", "email_domain", "sb_brand"}
APPS = {"TV", "NB", "MNT"}
ENTITY_ID_RE = r"^[a-z][a-z0-9_]*$"
DOMAIN_RE = r"^[a-z0-9-]+(\.[a-z0-9-]+)+$"

In [ ]:
# [c03] pure_seed_validate
def parse_csv_text(text: str, expect_cols: list[str]) -> list[dict]:
    """CSV 格式文字 → list[dict]；欄位順序必須等於 expect_cols，值去頭尾空白，空字串 → None。"""
    reader = csv.DictReader(io.StringIO(text.lstrip("\ufeff").strip()))
    if reader.fieldnames != expect_cols:
        raise ValueError(f"欄位應為 {expect_cols}，實際 {reader.fieldnames}")
    return [{k: (v.strip() or None) if isinstance(v, str) else v for k, v in r.items()} for r in reader]


def norm_alias(s: str) -> str:
    """比對用正規化：全形英數空白轉半形、小寫、空白壓成一格。Spark 端 alias_norm_col 必須同結果。"""
    s = "".join(chr(ord(c) - 0xFEE0) if "\uff01" <= c <= "\uff5e" else (" " if c == "\u3000" else c) for c in s)
    return re.sub(r"\s+", " ", s).strip().lower()


def validate_seed(entities: list[dict], aliases: list[dict]) -> list[str]:
    """回傳錯誤訊息；空 list = 通過。行號 = [c05] 該段第幾行（1 是欄名）。"""
    errs = []
    ids = {}
    for i, r in enumerate(entities, start=2):
        eid = r["entity_id"] or ""
        if not re.match(ENTITY_ID_RE, eid):
            errs.append(f"ENTITY_SEED 第 {i} 行 entity_id 格式不符（小寫英數底線）：{eid!r}")
        if eid in ids:
            errs.append(f"ENTITY_SEED 第 {i} 行 entity_id 重複（第 {ids[eid]} 行）：{eid}")
        ids[eid] = i
        if not r["entity_name"]:
            errs.append(f"ENTITY_SEED 第 {i} 行 entity_name 空白")
        if r["entity_type"] not in ENTITY_TYPES:
            errs.append(f"ENTITY_SEED 第 {i} 行 entity_type 不在 {sorted(ENTITY_TYPES)}：{r['entity_type']!r}")
        if r["side"] not in SIDES:
            errs.append(f"ENTITY_SEED 第 {i} 行 side 不在 {sorted(SIDES)}：{r['side']!r}")
        bad_apps = [a for a in (r["app_default"] or "").split("|") if a and a not in APPS]
        if bad_apps:
            errs.append(f"ENTITY_SEED 第 {i} 行 app_default 只能是 {sorted(APPS)} 以 | 分隔：{bad_apps}")
    seen = {}
    for i, r in enumerate(aliases, start=2):
        if r["alias_type"] not in ALIAS_TYPES:
            errs.append(f"ALIAS_SEED 第 {i} 行 alias_type 不在 {sorted(ALIAS_TYPES)}：{r['alias_type']!r}")
        if not r["alias"]:
            errs.append(f"ALIAS_SEED 第 {i} 行 alias 空白")
            continue
        if r["entity_id"] not in ids:
            errs.append(f"ALIAS_SEED 第 {i} 行 entity_id 不在 ENTITY_SEED：{r['entity_id']!r}")
        if r["alias_type"] == "email_domain" and not re.match(DOMAIN_RE, r["alias"]):
            errs.append(f"ALIAS_SEED 第 {i} 行 email_domain 應為小寫網域：{r['alias']!r}")
        key = (r["alias_type"], norm_alias(r["alias"]))
        if key in seen:
            errs.append(f"ALIAS_SEED 第 {i} 行 與第 {seen[key]} 行正規化後重複：{key}")
        seen[key] = i
    return errs

In [ ]:
# [c04] pure_seed_transform
FULLWIDTH = "".join(chr(c) for c in range(0xFF01, 0xFF5F)) + "\u3000"
HALFWIDTH = "".join(chr(c - 0xFEE0) for c in range(0xFF01, 0xFF5F)) + " "


def alias_norm_col(col):
    """與 norm_alias 同規則的 Spark 版本（T1-c / T2 查表時也用這個）。"""
    c = F.translate(F.coalesce(col, F.lit("")), FULLWIDTH, HALFWIDTH)
    return F.lower(F.trim(F.regexp_replace(c, r"\s+", " ")))


def transform_entity(df: DataFrame) -> DataFrame:
    """種子原始字串列 → s_semantic_entity 形狀（不含 updated_at）。"""
    apps = F.filter(F.split(F.coalesce("app_default", F.lit("")), r"\|"), lambda x: x != "")
    return df.select(
        "entity_id", "entity_name", "entity_type", "side",
        apps.alias("app_default"), "note", F.lit(True).alias("is_active"),
    )


def transform_alias(df: DataFrame) -> DataFrame:
    """種子原始字串列 → s_semantic_entity_alias 形狀（不含 updated_at）。"""
    return df.select(
        "alias_type", "alias", alias_norm_col(F.col("alias")).alias("alias_norm"),
        "entity_id", "note", F.lit(True).alias("is_active"),
    )

In [ ]:
# [c05] seed_data
# 對照表正本：改這裡 → 重跑本 notebook（先 dry_run）。格式同 CSV，第一行是欄名、逗號分隔、值不加引號。
# entity：entity_type = customer / self / supplier / partner / tool；side = auo / customer / third_party / ignore；
#         app_default = TV / NB / MNT，多個用 | 分隔。
# alias ：alias_type = name（素材原名）/ email_domain（小寫，後綴比對）/ sb_brand（Sales Brief brand_code）。
# note 標「待確認」的列是推測值，確認後改掉註記即可。
ENTITY_SEED = """
entity_id,entity_name,entity_type,side,app_default,note
auo,AUO,self,auo,,自家
auo_dplus,AUO Display Plus,self,auo,,auodplus.com；待確認是否視為自家
acer,Acer,customer,customer,MNT|NB,
asus,ASUS,customer,customer,MNT|NB,
dell,Dell,customer,customer,MNT|NB,Dell FY 自 2 月起算
hp,HP,customer,customer,MNT|NB,
lenovo,Lenovo,customer,customer,NB,
msi,MSI,customer,customer,MNT|NB,
tcl,TCL,customer,customer,TV,
hisense,Hisense,customer,customer,TV,
samsung_vd,Samsung VD,customer,customer,TV|MNT,samsung.com 網域也歸這裡；待確認是否要拆事業部
tpv,TPV,customer,customer,TV,
sony,SONY,customer,customer,TV,不在 Sales Brief 品牌清單
lg,LG,customer,customer,,Phase 0 新發現（lge.com 12 封）；app 待確認
benq,BenQ,customer,customer,,Phase 0 新發現；待確認與 Qisda 是否合併
qisda,Qisda,customer,customer,,Phase 0 新發現；待確認
inventec,Inventec,customer,customer,,Phase 0 新發現（ODM）；待確認
s_core,S-Core,partner,third_party,,Phase 0 新發現（s-core.co.kr）；身分待確認
sumika,Sumika,supplier,third_party,,Phase 0 新發現（sumika.com.tw）；待確認
mail_service,郵件 / 會議服務,tool,ignore,,webex / zoom / outlook / gmail 等，不當參與者公司
"""

ALIAS_SEED = """
alias_type,alias,entity_id,note
name,AUO,auo,
name,友達,auo,
name,Acer,acer,
name,宏碁,acer,
name,ASUS,asus,
name,華碩,asus,
name,Dell,dell,
name,HP,hp,
name,Lenovo,lenovo,
name,聯想,lenovo,
name,MSI,msi,
name,微星,msi,
name,TCL,tcl,
name,Hisense,hisense,
name,海信,hisense,
name,Samsung VD,samsung_vd,
name,VD,samsung_vd,
name,Samsung,samsung_vd,待確認：非 VD 事業部是否另立
name,三星,samsung_vd,
name,TPV,tpv,
name,冠捷,tpv,
name,SONY,sony,
name,LG,lg,
name,LGE,lg,
name,BenQ,benq,
name,明基,benq,
name,Qisda,qisda,
name,佳世達,qisda,
name,Inventec,inventec,
name,英業達,inventec,
name,Sumika,sumika,
name,住華,sumika,
email_domain,auo.com,auo,後綴比對，含 ab1.corpnet.auo.com
email_domain,auocorporation.onmicrosoft.com,auo,
email_domain,auodplus.com,auo_dplus,
email_domain,acer.com,acer,
email_domain,acer.mail.onmicrosoft.com,acer,
email_domain,acer.onmicrosoft.com,acer,
email_domain,asus.com,asus,
email_domain,dell.com,dell,
email_domain,hp.com,hp,
email_domain,lenovo.com,lenovo,
email_domain,msi.com,msi,
email_domain,tcl.com,tcl,Phase 0 未出現，依規劃初表
email_domain,hisense.com,hisense,Phase 0 未出現，依規劃初表
email_domain,samsung.com,samsung_vd,
email_domain,tpv-tech.com,tpv,Phase 0 未出現，依規劃初表
email_domain,sony.com,sony,Phase 0 未出現，依規劃初表
email_domain,lge.com,lg,
email_domain,benq.com,benq,
email_domain,qisda.com,qisda,
email_domain,inventec.com,inventec,
email_domain,s-core.co.kr,s_core,
email_domain,sumika.com.tw,sumika,
email_domain,webex.com,mail_service,
email_domain,zoomcrc.com,mail_service,
email_domain,outlook.com,mail_service,後綴比對，含 *.prod.outlook.com
email_domain,gmail.com,mail_service,個人信箱，公司不明
sb_brand,Acer,acer,Sales Brief brand_code 實際值（Phase 0）
sb_brand,Asus,asus,
sb_brand,Dell,dell,
sb_brand,HP,hp,
sb_brand,Lenovo,lenovo,
sb_brand,MSI,msi,
sb_brand,Hisense,hisense,
sb_brand,Samsung,samsung_vd,TV 與 MNT 的 brand_code 都是 Samsung
sb_brand,TCL,tcl,
sb_brand,TPV,tpv,
"""

In [ ]:
# [c10] load_seed
entity_rows = parse_csv_text(ENTITY_SEED, ENTITY_COLS)
alias_rows = parse_csv_text(ALIAS_SEED, ALIAS_COLS)
errors = validate_seed(entity_rows, alias_rows)
for e in errors:
    print("ERROR", e)
assert not errors, f"種子驗證失敗 {len(errors)} 筆，修 [c05] 後重跑"

str_schema = lambda cols: ", ".join(f"{c} string" for c in cols)  # noqa: E731
df_entity = transform_entity(spark.createDataFrame(entity_rows, str_schema(ENTITY_COLS)))
df_alias = transform_alias(spark.createDataFrame(alias_rows, str_schema(ALIAS_COLS)))
print(f"entity={len(entity_rows)} alias={len(alias_rows)}",
      {t: sum(r["alias_type"] == t for r in alias_rows) for t in sorted(ALIAS_TYPES)})

In [ ]:
# [c11] preview_diff
# 與現有表比較：新增 / 內容變更 / 將停用（[c05] 已拿掉）。表不存在時全部算新增。
fq_entity = f"{catalog}.{schema}.s_semantic_entity"
fq_alias = f"{catalog}.{schema}.s_semantic_entity_alias"
fq_log = f"{catalog}.{schema}.b_meeting_migration_log"


def diff(df_new: DataFrame, fq: str, keys: list[str], cmp: list[str]) -> dict:
    if not spark.catalog.tableExists(fq):
        return {"insert": df_new.count(), "update": 0, "deactivate": 0}
    old = spark.table(fq)
    j = df_new.alias("n").join(old.alias("o"), keys, "full_outer")
    is_new = F.col(f"o.{cmp[0]}").isNull() & F.col(f"n.{cmp[0]}").isNotNull()
    gone = F.col(f"n.{cmp[0]}").isNull() & F.col("o.is_active")
    same = F.lit(True)
    for c in cmp:
        same = same & F.col(f"n.{c}").eqNullSafe(F.col(f"o.{c}"))
    changed = F.col(f"n.{cmp[0]}").isNotNull() & F.col(f"o.{cmp[0]}").isNotNull() & ~same
    r = j.agg(*[F.sum(x.cast("int")).alias(k) for k, x in
                {"insert": is_new, "update": changed, "deactivate": gone}.items()]).collect()[0]
    return {k: int(r[k] or 0) for k in ("insert", "update", "deactivate")}


ENTITY_CMP = ["entity_name", "entity_type", "side", "app_default", "note", "is_active"]
ALIAS_CMP = ["alias", "entity_id", "note", "is_active"]
result = {
    "entity": diff(df_entity, fq_entity, ["entity_id"], ENTITY_CMP),
    "alias": diff(df_alias, fq_alias, ["alias_type", "alias_norm"], ALIAS_CMP),
}
print("預計變更：", result)
if dry_run:
    print("dry_run = true：不建表、不寫入。確認上面的變更後把 dry_run 改 false 重跑。")

In [ ]:
# [c20] create_tables
# 一次性 DDL；IF NOT EXISTS 讓重跑無害。表很小，不設 partition / cluster。
DDL = [
    f"""CREATE TABLE IF NOT EXISTS {fq_entity} (
        entity_id STRING NOT NULL COMMENT '公司鍵，小寫英數底線（例 samsung_vd）',
        entity_name STRING COMMENT '正規名稱（例 Samsung VD）',
        entity_type STRING COMMENT 'customer / self / supplier / partner / tool',
        side STRING COMMENT '參與者發言方：auo / customer / third_party / ignore（tool 類不當參與者公司）',
        app_default ARRAY<STRING> COMMENT '客戶預設應用（TV / NB / MNT），會議條目 app 推不出時用',
        note STRING,
        is_active BOOLEAN COMMENT '種子拿掉的列標 false，不刪',
        updated_at TIMESTAMP
    ) COMMENT '公司主檔。正本：001_semantic_entity [c05] ENTITY_SEED'""",
    f"""CREATE TABLE IF NOT EXISTS {fq_alias} (
        alias_type STRING NOT NULL COMMENT 'name（素材原名）/ email_domain（後綴比對）/ sb_brand（Sales Brief brand_code）',
        alias STRING COMMENT '原字串',
        alias_norm STRING NOT NULL COMMENT '比對鍵：全形轉半形、小寫、空白壓縮',
        entity_id STRING COMMENT 'FK → s_semantic_entity.entity_id',
        note STRING,
        is_active BOOLEAN,
        updated_at TIMESTAMP
    ) COMMENT '公司別名。正本：001_semantic_entity [c05] ALIAS_SEED'""",
    f"""CREATE TABLE IF NOT EXISTS {fq_log} (
        slug STRING NOT NULL COMMENT 'migration 流水號＋名稱（例 001_semantic_entity）',
        run_at TIMESTAMP NOT NULL,
        run_by STRING,
        result STRING COMMENT 'JSON：各表 insert / update / deactivate 筆數',
        note STRING
    ) COMMENT '會議萃取 migration 執行紀錄（只增不改）'""",
]
if not dry_run:
    for ddl in DDL:
        spark.sql(ddl)
    print("tables ready")

In [ ]:
# [c21] merge_seed
# MERGE：鍵相同且內容有變才更新 updated_at；種子拿掉的列標 is_active = false（不刪，保留回溯）。
def merge(df_new: DataFrame, fq: str, keys: list[str], cmp: list[str]) -> None:
    view = f"_seed_{fq.split('.')[-1]}"
    df_new.createOrReplaceTempView(view)
    on = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    differs = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in cmp)
    cols = keys + [c for c in cmp if c not in keys]
    spark.sql(f"""
        MERGE INTO {fq} t USING {view} s ON {on}
        WHEN MATCHED AND ({differs}) THEN UPDATE SET
            {", ".join(f"t.{c} = s.{c}" for c in cmp)}, t.updated_at = current_timestamp()
        WHEN NOT MATCHED THEN INSERT ({", ".join(cols)}, updated_at)
            VALUES ({", ".join(f"s.{c}" for c in cols)}, current_timestamp())
        WHEN NOT MATCHED BY SOURCE AND t.is_active THEN UPDATE SET
            t.is_active = false, t.updated_at = current_timestamp()
    """)


if not dry_run:
    merge(df_entity, fq_entity, ["entity_id"], ENTITY_CMP)
    merge(df_alias, fq_alias, ["alias_type", "alias_norm"], ALIAS_CMP)
    spark.sql(
        f"INSERT INTO {fq_log} SELECT :slug, current_timestamp(), current_user(), :result, :note",
        args={"slug": SLUG, "result": json.dumps(result), "note": None},
    )
    print("merged", result)

In [ ]:
# [c30] check_target
if not dry_run:
    ent = spark.table(fq_entity)
    ali = spark.table(fq_alias)
    dup_e = ent.groupBy("entity_id").count().where("count > 1").count()
    dup_a = ali.groupBy("alias_type", "alias_norm").count().where("count > 1").count()
    orphan = (ali.where("is_active").join(ent.where("is_active"), "entity_id", "left_anti").count())
    print(f"entity active={ent.where('is_active').count()} alias active={ali.where('is_active').count()} "
          f"dup_entity={dup_e} dup_alias={dup_a} orphan_alias={orphan}")
    ali.where("is_active").groupBy("alias_type").count().orderBy("alias_type").show()
    assert dup_e == 0 and dup_a == 0, "鍵重複"
    assert orphan == 0, "有 alias 指向不存在或已停用的 entity"